# Width-Scaling SGD Experiments, d8/d16 Excluded

Train parity networks with d8 and d16 targets excluded at widths `N = 256, 512, 1024, 2048` with SGD, save train/test curves, and analyze the final checkpoints with PCA rank reduction and weight-variance scaling.


## Setup

Mount Google Drive, clone the public repo, install it in editable mode, and define the run/plot directories.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [1]:
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = Path("/content/feature-learning-parity-task")

# DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_unfrozen_emb_mup")
# DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_exclude_d4_d8_d16_8_rel_bits")
DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_exclude_d4_d8_d16_1-layer")
DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_exclude_d8_d16_hierarchy")


RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!pip install -e .


## Train Width Sweep

This cell writes one config per width for a task with d8 and d16 targets excluded, then runs training inside the notebook kernel so the progress bar renders in Colab. Existing final checkpoints are skipped unless `FORCE_RETRAIN = True`.


In [ ]:
from pathlib import Path

import yaml

from parity_net.config import load_config
from parity_net.train import train

WIDTHS = [1024, 2048]
FORCE_RETRAIN = True


def make_config(N: int, output_dir: Path) -> dict:
    return {
        "model": {
            "input_dim": 32,
            "relevant_dim": 16,
            "N": N,
            "L": 1,
            "activation": "half-tanh",
            "use_readout_barrier": False,
            "embedding_weight_variance": 1.0 / 32,
            "freeze_embedding": False,
            "hidden_weight_variance": 1.0 / N,
            "readout_weight_variance": 1.0 / N,
            "use_layerwise_readouts": True,
            "use_post_activation_linear": False,
            "bias": False,
        },
        "task": {
            "input_dim": 32,
            "relevant_dim": 16,
            "exclude_targets": ["d4", "d8", "d16"],
        },
        "training": {
            "num_steps": 10000,
            "test_samples": 100_000,
            "batch_size": 512,
            "seed": 0,
            "device": "cuda",
            "dtype": "float32",
            "log_every": 1_000,
            "checkpoint_every": 1_000,
            "output_dir": str(output_dir),
            "barrier_c": None,
            "barrier_lambda": 10.0,
            "optimizer": {
                "name": "sgd",
                "lr": 1e-2,
                "lr_embedding": None,
                "lr_hidden": None,
                "lr_readout": None,
                "weight_decay": 1e-3,
                "wd_embedding": None,
                "wd_hidden": None,
                "wd_readout": None,
                "momentum": 0.9,
                "betas": [0.9, 0.999],
            },
        },
    }


def make_config_mup(N: int, output_dir: Path) -> dict:
    config = make_config(N, output_dir)
    optimizer = config["training"]["optimizer"]
    base_lr = optimizer["lr"]
    base_wd = optimizer["weight_decay"]
    config["model"]["readout_weight_variance"] = 1 / N**2
    optimizer["lr_embedding"] = base_lr * N / 256
    optimizer["lr_hidden"] = base_lr
    optimizer["lr_readout"] = base_lr * 256 / N
    optimizer["wd_embedding"] = base_wd * 256 / N
    optimizer["wd_hidden"] = base_wd
    optimizer["wd_readout"] = base_wd * N / 256
    return config


CONFIG_FACTORY = make_config_mup  # Change to make_config_mup for the muP-scaled sweep.


In [ ]:
config_paths = {}
for N in WIDTHS:
    run_dir = RUNS_DIR / f"N_{N}"
    run_dir.mkdir(parents=True, exist_ok=True)
    config_path = run_dir / "config.yaml"
    final_checkpoint = run_dir / "checkpoints" / "final.pt"

    if final_checkpoint.exists() and not FORCE_RETRAIN:
        config_paths[N] = config_path
        print(
            f"N={N}: final checkpoint exists, skipping training: {final_checkpoint}. "
            "Set FORCE_RETRAIN=True or choose a new DRIVE_ROOT to train with changed config values."
        )
        continue

    config = CONFIG_FACTORY(N, run_dir)
    with config_path.open("w") as f:
        yaml.safe_dump(config, f, sort_keys=False)
    config_paths[N] = config_path

    print(f"N={N}: training with {config_path}")
    final_path = train(load_config(config_path))
    print(f"N={N}: saved final checkpoint to {final_path}")


## Train/Test Curves

Read `metrics.csv` from each run, save one plot per width, and save combined train/test plots across widths.


In [ ]:
# DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_unfrozen_emb")
# DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd")

RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

WIDTHS = [1024, 2048]


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

USE_LOG_MSE_AXIS = True
USE_LOG_STEP_AXIS = True
TEST_MSE_COLUMNS = ["test_mse", "test_mse_d2", "test_mse_d4", "test_mse_d8", "test_mse_d16"]


def axis_has_positive_data(ax, axis):
    for line in ax.lines:
        values = line.get_xdata() if axis == "x" else line.get_ydata()
        if len(values) and pd.Series(values).dropna().gt(0).any():
            return True
    return False


def maybe_log_y(ax):
    if USE_LOG_MSE_AXIS and axis_has_positive_data(ax, "y"):
        ax.set_yscale("log")


def maybe_log_x(ax):
    if USE_LOG_STEP_AXIS and axis_has_positive_data(ax, "x"):
        ax.set_xscale("log")


metrics_by_width = {}
for N in WIDTHS:
    metrics_path = RUNS_DIR / f"N_{N}" / "metrics.csv"
    if not metrics_path.exists():
        print(f"N={N}: missing {metrics_path}")
        continue
    df = pd.read_csv(metrics_path)
    metrics_by_width[N] = df

    fig, ax = plt.subplots(figsize=(8, 5))
    for column in TEST_MSE_COLUMNS:
        if column in df.columns:
            ax.plot(df["step"], df[column], label=column)
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE")
    ax.set_title(f"Test MSE by degree, N={N}")
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"test_mse_by_degree_N_{N}.png", dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["step"], df["train_mse"], label="train_mse")
    ax.plot(df["step"], df["test_mse"], label="test_mse")
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE")
    ax.set_title(f"Train/Test MSE, N={N}")
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"train_test_mse_N_{N}.png", dpi=150)
    plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for N, df in metrics_by_width.items():
    ax.plot(df["step"], df["test_mse"], label=f"N={N}")
ax.set_xlabel("Step")
ax.set_ylabel("Test MSE")
ax.set_title("Test MSE vs Step")
maybe_log_x(ax)
maybe_log_y(ax)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "test_mse_by_width.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for N, df in metrics_by_width.items():
    ax.plot(df["step"], df["train_mse"], label=f"N={N}")
ax.set_xlabel("Step")
ax.set_ylabel("Train MSE")
ax.set_title("Train MSE vs Step")
maybe_log_x(ax)
maybe_log_y(ax)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "train_mse_by_width.png", dpi=150)
plt.show()


## Final-Checkpoint PCA and Weight-Variance Analysis

For each final checkpoint, load the saved test set, compute PCA ranks at the selected residual-stream layer, sweep PCA interventions, and collect trained weight variances.


In [ ]:
import torch

from parity_net.analysis import (
    collect_layer_activations,
    make_pca_intervention,
    pca_from_activations,
    per_degree_mse,
    predict_in_batches,
    rank_for_threshold,
)
from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset, target_names
from parity_net.train import resolve_device, resolve_dtype

PCA_SAMPLES = 20_000
CHECKPOINT_STEP = "final"  # Use "final" or an integer step, e.g. 50000.
USE_LOG_PCA_MSE_AXIS = True
ANALYSIS_LAYER_IDX = 1  # 0 is embedding, later indices are after residual blocks.
KEEP_PCS_MIN = 0
KEEP_PCS_MAX = 50
KEEP_PCS_STEP = 1


def checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('CHECKPOINT_STEP must be "final" or an integer step')


def assert_finite_tensor(tensor, label):
    if not torch.isfinite(tensor).all().item():
        raise ValueError(f"{label} contains NaN or Inf")


def assert_finite_metrics(metrics, label):
    bad = {key: value for key, value in metrics.items() if not pd.notna(value)}
    if bad:
        raise ValueError(f"{label} contains non-finite values: {bad}")


checkpoint_file, checkpoint_label = checkpoint_name(CHECKPOINT_STEP)
pca_mse_scale_label = "mse_log_scale" if USE_LOG_PCA_MSE_AXIS else "mse_linear_scale"
rank_rows = []
weight_rows = []
pca_sweep_by_width = {}
skipped_widths = []

for N in WIDTHS:
    run_dir = RUNS_DIR / f"N_{N}"
    checkpoint_path = run_dir / "checkpoints" / checkpoint_file
    if not checkpoint_path.exists():
        print(f"N={N}: missing checkpoint {checkpoint_path}, skipping analysis")
        skipped_widths.append({"N": N, "reason": "missing checkpoint"})
        continue

    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, payload, _ = load_checkpoint(checkpoint_path, device)
        training = payload["config"]["training"]
        model_config = payload["config"]["model"]
        task_config = payload["config"].get("task") or {
            "input_dim": model_config["input_dim"],
            "relevant_dim": model_config["relevant_dim"],
            "exclude_targets": [],
        }
        target_names_ = target_names(
            int(task_config["relevant_dim"]),
            list(task_config.get("exclude_targets", [])),
        )
        device = resolve_device(training["device"])
        dtype = resolve_dtype(training["dtype"])
        model = model.to(device=device, dtype=dtype)
        batch_size = training["batch_size"]

        test_data_path = Path(payload.get("test_data_path") or run_dir / "test_data.pt")
        test_data = load_dataset(test_data_path, device, dtype)
        if PCA_SAMPLES < test_data.x.shape[0]:
            heldout = ParityDataset(x=test_data.x[:PCA_SAMPLES], y=test_data.y[:PCA_SAMPLES])
        else:
            heldout = test_data

        local_weight_rows = []
        for layer, variance in model.weight_variances().items():
            if not pd.notna(variance):
                raise ValueError(f"weight variance for {layer} is non-finite: {variance}")
            local_weight_rows.append({"N": N, "checkpoint": checkpoint_label, "layer": layer, "variance": variance})

        print(f"N={N}: collecting activations from {heldout.x.shape[0]} saved test samples")
        activations = collect_layer_activations(model, heldout.x, batch_size)
        for layer_idx, layer_acts in enumerate(activations):
            assert_finite_tensor(layer_acts, f"N={N} activations layer {layer_idx}")

        pcas = [pca_from_activations(layer_acts) for layer_acts in activations]
        for layer_idx, pca in enumerate(pcas):
            assert_finite_tensor(pca["cumulative_explained_variance"], f"N={N} PCA cumulative layer {layer_idx}")
            assert_finite_tensor(pca["components"], f"N={N} PCA components layer {layer_idx}")

        local_rank_rows = []
        for layer_idx, pca in enumerate(pcas):
            cumulative = pca["cumulative_explained_variance"]
            local_rank_rows.append(
                {
                    "N": N,
                    "checkpoint": checkpoint_label,
                    "layer_idx": layer_idx,
                    "rank_90": rank_for_threshold(cumulative, 0.90),
                    "rank_99": rank_for_threshold(cumulative, 0.99),
                    "num_dimensions": cumulative.numel(),
                }
            )

        if ANALYSIS_LAYER_IDX < 0 or ANALYSIS_LAYER_IDX >= len(pcas):
            raise ValueError(f"ANALYSIS_LAYER_IDX must be in [0, {len(pcas) - 1}]")

        max_available_pcs = pcas[ANALYSIS_LAYER_IDX]["components"].shape[0]
        keep_pcs_max_effective = min(KEEP_PCS_MAX, max_available_pcs)
        keep_pcs_values = list(range(KEEP_PCS_MIN, keep_pcs_max_effective + 1, KEEP_PCS_STEP))
        if keep_pcs_values[-1] != keep_pcs_max_effective:
            keep_pcs_values.append(keep_pcs_max_effective)

        sweep_rows = []
        for keep_pcs in keep_pcs_values:
            intervention = make_pca_intervention(pcas[ANALYSIS_LAYER_IDX], keep_pcs)
            with torch.no_grad():
                pred_intervened = predict_in_batches(
                    model,
                    heldout.x,
                    batch_size,
                    intervention=(ANALYSIS_LAYER_IDX, intervention),
                )
            assert_finite_tensor(pred_intervened, f"N={N} intervened predictions keep_pcs={keep_pcs}")
            metrics = per_degree_mse(pred_intervened, heldout.y, target_names_)
            assert_finite_metrics(metrics, f"N={N} MSE metrics keep_pcs={keep_pcs}")
            sweep_rows.append(
                {
                    "N": N,
                    "checkpoint": checkpoint_label,
                    "intervention_layer": ANALYSIS_LAYER_IDX,
                    "keep_pcs": keep_pcs,
                    **metrics,
                }
            )

        sweep_df = pd.DataFrame(sweep_rows)
        pca_sweep_by_width[N] = sweep_df
        sweep_df.to_csv(ANALYSIS_DIR / f"pca_mse_sweep_{checkpoint_label}_N_{N}.csv", index=False)

        fig, ax = plt.subplots(figsize=(8, 5))
        pca_metric_columns = [
            column
            for column in ["mse_all", "mse_d2", "mse_d4", "mse_d8", "mse_d16"]
            if column in sweep_df.columns
        ]
        sweep_df.plot(
            x="keep_pcs",
            y=pca_metric_columns,
            marker="o",
            ax=ax,
        )
        ax.set_xlabel("Number of PCs kept")
        ax.set_ylabel("MSE")
        ax.set_title(f"PCA intervention MSE, N={N}, {checkpoint_label}, layer={ANALYSIS_LAYER_IDX}")
        if USE_LOG_PCA_MSE_AXIS:
            ax.set_yscale("log")
        ax.grid(True, alpha=0.3, which="both")
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / f"pca_{pca_mse_scale_label}_sweep_{checkpoint_label}_layer_{ANALYSIS_LAYER_IDX}_N_{N}.png", dpi=150)
        plt.show()

        weight_rows.extend(local_weight_rows)
        rank_rows.extend(local_rank_rows)

    except Exception as exc:
        print(f"N={N}: checkpoint {checkpoint_path} is broken or non-finite, skipping. Reason: {exc}")
        skipped_widths.append({"N": N, "reason": str(exc)})
        plt.close("all")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

rank_df = pd.DataFrame(rank_rows)
weight_variance_df = pd.DataFrame(weight_rows)
skipped_widths_df = pd.DataFrame(skipped_widths)
rank_df.to_csv(ANALYSIS_DIR / f"pca_ranks_by_width_{checkpoint_label}.csv", index=False)
weight_variance_df.to_csv(ANALYSIS_DIR / f"weight_variances_by_width_{checkpoint_label}.csv", index=False)
skipped_widths_df.to_csv(ANALYSIS_DIR / f"skipped_widths_{checkpoint_label}.csv", index=False)

display(rank_df)
display(weight_variance_df)
if not skipped_widths_df.empty:
    print("Skipped widths")
    display(skipped_widths_df)


PCA for activations after each residual block's nonlinearity, before the residual addition. This also sweeps the number of PCs kept at one chosen block and measures test MSE after continuing the forward pass with the projected nonlinear activation.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

from parity_net.analysis import (
    make_pca_intervention,
    pca_from_activations,
    per_degree_mse,
    predict_in_batches,
    rank_for_threshold,
)
from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset, target_names
from parity_net.train import resolve_device, resolve_dtype

PRE_ADD_PCA_SAMPLES = PCA_SAMPLES
PRE_ADD_CHECKPOINT_STEP = CHECKPOINT_STEP  # Use "final" or an integer step, e.g. 50000.
PRE_ADD_ANALYSIS_BLOCK_IDX = max(ANALYSIS_LAYER_IDX - 1, 0)  # Block whose phi(Wx) gets PCA-projected.
PRE_ADD_KEEP_PCS_MIN = KEEP_PCS_MIN
PRE_ADD_KEEP_PCS_MAX = KEEP_PCS_MAX
PRE_ADD_KEEP_PCS_STEP = KEEP_PCS_STEP
PRE_ADD_USE_LOG_MSE_AXIS = USE_LOG_PCA_MSE_AXIS


def pre_add_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('PRE_ADD_CHECKPOINT_STEP must be "final" or an integer step')


@torch.no_grad()
def collect_pre_add_nonlinearity_activations(model, x, batch_size):
    all_layers = [[] for _ in model.blocks]
    model.eval()
    for start in range(0, x.shape[0], batch_size):
        stop = min(start + batch_size, x.shape[0])
        h = model.embedding(x[start:stop])
        for layer_idx, block in enumerate(model.blocks):
            update = block.activation(block.linear(h))
            all_layers[layer_idx].append(update.detach())
            if block.post_activation_linear is not None:
                update_for_residual = block.post_activation_linear(update)
            else:
                update_for_residual = update
            h = h + update_for_residual
    return [torch.cat(chunks, dim=0) for chunks in all_layers]


def individual_target_mse(pred, y, target_names_):
    return {
        f"mse_{target_name}": F.mse_loss(pred[:, target_idx], y[:, target_idx]).item()
        for target_idx, target_name in enumerate(target_names_)
    }


pre_add_checkpoint_file, pre_add_checkpoint_label = pre_add_checkpoint_name(PRE_ADD_CHECKPOINT_STEP)
pre_add_mse_scale_label = "mse_log_scale" if PRE_ADD_USE_LOG_MSE_AXIS else "mse_linear_scale"
pre_add_rank_rows = []
pre_add_sweep_by_width = {}
pre_add_skipped_widths = []
pre_add_pcas_by_width = {}

for N in WIDTHS:
    run_dir = RUNS_DIR / f"N_{N}"
    checkpoint_path = run_dir / "checkpoints" / pre_add_checkpoint_file
    if not checkpoint_path.exists():
        print(f"N={N}: missing checkpoint {checkpoint_path}, skipping pre-addition PCA")
        pre_add_skipped_widths.append({"N": N, "reason": "missing checkpoint"})
        continue

    try:
        load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, payload, _ = load_checkpoint(checkpoint_path, load_device)
        training = payload["config"]["training"]
        model_config = payload["config"]["model"]
        task_config = payload["config"].get("task") or {
            "input_dim": model_config["input_dim"],
            "relevant_dim": model_config["relevant_dim"],
            "exclude_targets": [],
        }
        target_names_ = target_names(
            int(task_config["relevant_dim"]),
            list(task_config.get("exclude_targets", [])),
        )
        d2_target_names = [name for name in target_names_ if name.startswith("d2_")]
        if len(d2_target_names) != len(target_names_):
            print(
                f"N={N}: task contains non-d2 targets too; individual MSE columns will include all targets, "
                "and mse_d2 is the d2 average."
            )

        device = resolve_device(training["device"])
        dtype = resolve_dtype(training["dtype"])
        model = model.to(device=device, dtype=dtype).eval()
        batch_size = int(training["batch_size"])

        test_data_path = Path(payload.get("test_data_path") or run_dir / "test_data.pt")
        test_data = load_dataset(test_data_path, device, dtype)
        if PRE_ADD_PCA_SAMPLES is not None and PRE_ADD_PCA_SAMPLES < test_data.x.shape[0]:
            heldout = ParityDataset(x=test_data.x[:PRE_ADD_PCA_SAMPLES], y=test_data.y[:PRE_ADD_PCA_SAMPLES])
        elif PRE_ADD_PCA_SAMPLES is not None and PRE_ADD_PCA_SAMPLES > test_data.x.shape[0]:
            raise ValueError(
                f"Requested PRE_ADD_PCA_SAMPLES={PRE_ADD_PCA_SAMPLES}, but saved test set has "
                f"{test_data.x.shape[0]} samples"
            )
        else:
            heldout = test_data

        print(
            f"N={N}: collecting post-nonlinearity/pre-addition activations "
            f"from {heldout.x.shape[0]} saved test samples"
        )
        pre_add_activations = collect_pre_add_nonlinearity_activations(model, heldout.x, batch_size)
        if PRE_ADD_ANALYSIS_BLOCK_IDX < 0 or PRE_ADD_ANALYSIS_BLOCK_IDX >= len(pre_add_activations):
            raise ValueError(
                f"PRE_ADD_ANALYSIS_BLOCK_IDX must be in [0, {len(pre_add_activations) - 1}]"
            )
        for block_idx, layer_acts in enumerate(pre_add_activations):
            if not torch.isfinite(layer_acts).all().item():
                raise ValueError(f"pre-addition activations contain NaN or Inf at block {block_idx}")

        pre_add_pcas = [pca_from_activations(layer_acts) for layer_acts in pre_add_activations]
        pre_add_pcas_by_width[N] = pre_add_pcas
        for block_idx, pca in enumerate(pre_add_pcas):
            cumulative = pca["cumulative_explained_variance"]
            if not torch.isfinite(cumulative).all().item():
                raise ValueError(f"pre-addition PCA cumulative variance is non-finite at block {block_idx}")
            pre_add_rank_rows.append(
                {
                    "N": N,
                    "checkpoint": pre_add_checkpoint_label,
                    "block_idx": block_idx,
                    "activation": "after_nonlinearity_before_residual_addition",
                    "rank_90": rank_for_threshold(cumulative, 0.90),
                    "rank_99": rank_for_threshold(cumulative, 0.99),
                    "num_dimensions": cumulative.numel(),
                }
            )

        selected_pca = pre_add_pcas[PRE_ADD_ANALYSIS_BLOCK_IDX]
        max_available_pcs = selected_pca["components"].shape[0]
        keep_pcs_max_effective = min(PRE_ADD_KEEP_PCS_MAX, max_available_pcs)
        keep_pcs_values = list(
            range(PRE_ADD_KEEP_PCS_MIN, keep_pcs_max_effective + 1, PRE_ADD_KEEP_PCS_STEP)
        )
        if keep_pcs_values[-1] != keep_pcs_max_effective:
            keep_pcs_values.append(keep_pcs_max_effective)

        sweep_rows = []
        for keep_pcs in keep_pcs_values:
            block_intervention = make_pca_intervention(selected_pca, keep_pcs)
            with torch.no_grad():
                pred_intervened = predict_in_batches(
                    model,
                    heldout.x,
                    batch_size,
                    block_intervention=(PRE_ADD_ANALYSIS_BLOCK_IDX, block_intervention),
                )
            if not torch.isfinite(pred_intervened).all().item():
                raise ValueError(f"non-finite predictions for keep_pcs={keep_pcs}")
            metrics = per_degree_mse(pred_intervened, heldout.y, target_names_)
            per_target_metrics = individual_target_mse(pred_intervened, heldout.y, target_names_)
            sweep_rows.append(
                {
                    "N": N,
                    "checkpoint": pre_add_checkpoint_label,
                    "block_idx": PRE_ADD_ANALYSIS_BLOCK_IDX,
                    "activation": "after_nonlinearity_before_residual_addition",
                    "keep_pcs": keep_pcs,
                    **metrics,
                    **per_target_metrics,
                }
            )

        sweep_df = pd.DataFrame(sweep_rows)
        pre_add_sweep_by_width[N] = sweep_df
        sweep_path = (
            ANALYSIS_DIR
            / f"pre_add_nonlinearity_pca_mse_sweep_{pre_add_checkpoint_label}_block_{PRE_ADD_ANALYSIS_BLOCK_IDX}_N_{N}.csv"
        )
        sweep_df.to_csv(sweep_path, index=False)

        fig, ax = plt.subplots(figsize=(8, 5))
        mse_columns = ["mse_all", "mse_d2"] + [f"mse_{name}" for name in d2_target_names]
        mse_columns = [column for column in mse_columns if column in sweep_df.columns]
        sweep_df.plot(x="keep_pcs", y=mse_columns, marker="o", ax=ax)
        ax.set_xlabel("Number of PCs kept")
        ax.set_ylabel("MSE")
        ax.set_title(
            f"Pre-addition nonlinear PCA MSE, N={N}, {pre_add_checkpoint_label}, "
            f"block={PRE_ADD_ANALYSIS_BLOCK_IDX}"
        )
        if PRE_ADD_USE_LOG_MSE_AXIS:
            positive_values = sweep_df[mse_columns].to_numpy().reshape(-1)
            positive_values = positive_values[positive_values > 0]
            if len(positive_values):
                ax.set_yscale("log")
            else:
                print(f"N={N}: no positive MSE values, leaving pre-addition plot on linear scale")
        ax.grid(True, alpha=0.3, which="both")
        fig.tight_layout()
        fig.savefig(
            PLOTS_DIR
            / f"pre_add_nonlinearity_pca_{pre_add_mse_scale_label}_sweep_{pre_add_checkpoint_label}_block_{PRE_ADD_ANALYSIS_BLOCK_IDX}_N_{N}.png",
            dpi=150,
        )
        plt.show()
    except Exception as exc:
        print(f"N={N}: skipping pre-addition PCA. Reason: {exc}")
        pre_add_skipped_widths.append({"N": N, "reason": str(exc)})
        plt.close("all")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

pre_add_rank_df = pd.DataFrame(pre_add_rank_rows)
pre_add_skipped_widths_df = pd.DataFrame(pre_add_skipped_widths)
pre_add_rank_path = ANALYSIS_DIR / f"pre_add_nonlinearity_pca_ranks_by_width_{pre_add_checkpoint_label}.csv"
pre_add_skipped_path = ANALYSIS_DIR / f"pre_add_nonlinearity_skipped_widths_{pre_add_checkpoint_label}.csv"
pre_add_rank_df.to_csv(pre_add_rank_path, index=False)
pre_add_skipped_widths_df.to_csv(pre_add_skipped_path, index=False)

print(f"Saved pre-addition PCA ranks to {pre_add_rank_path}")
display(pre_add_rank_df)
if not pre_add_skipped_widths_df.empty:
    print("Skipped widths")
    display(pre_add_skipped_widths_df)


## Summary Plots

Plot effective residual-stream dimension versus width, PCA intervention curves across widths, and trained weight variance scaling versus width.


In [ ]:
SUMMARY_LAYER_IDX = 4
pca_mse_scale_label = "mse_log_scale" if USE_LOG_PCA_MSE_AXIS else "mse_linear_scale"

layer_rank_df = rank_df[rank_df["layer_idx"] == SUMMARY_LAYER_IDX].copy()
if layer_rank_df.empty:
    raise ValueError(f"No PCA rank rows found for SUMMARY_LAYER_IDX={SUMMARY_LAYER_IDX}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(layer_rank_df["N"], layer_rank_df["rank_90"], marker="o", label="90% variance")
ax.plot(layer_rank_df["N"], layer_rank_df["rank_99"], marker="o", label="99% variance")
ax.set_xscale("log", base=2)
ax.set_xlabel("Width N")
ax.set_ylabel("Number of PCs")
ax.set_title(f"Effective residual-stream dimension at layer {SUMMARY_LAYER_IDX}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / f"effective_dimension_vs_width_layer_{SUMMARY_LAYER_IDX}.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
plotted_pca_sweeps = 0
for N, sweep_df in pca_sweep_by_width.items():
    if "intervention_layer" in sweep_df:
        sweep_df = sweep_df[sweep_df["intervention_layer"] == SUMMARY_LAYER_IDX]
    if sweep_df.empty:
        continue
    ax.plot(sweep_df["keep_pcs"], sweep_df["mse_all"], marker="o", label=f"N={N}")
    plotted_pca_sweeps += 1
ax.set_xlabel("Number of PCs kept")
ax.set_ylabel("MSE, all targets")
ax.set_title(f"PCA intervention MSE across widths, layer {SUMMARY_LAYER_IDX}")
if USE_LOG_PCA_MSE_AXIS:
    ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
if plotted_pca_sweeps:
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"pca_{pca_mse_scale_label}_all_sweep_by_width_layer_{SUMMARY_LAYER_IDX}.png", dpi=150)
    plt.show()
else:
    plt.close(fig)
    print(
        f"No PCA intervention sweeps found for SUMMARY_LAYER_IDX={SUMMARY_LAYER_IDX}. "
        "Rerun the PCA analysis cell with ANALYSIS_LAYER_IDX set to this layer first."
    )

fig, ax = plt.subplots(figsize=(8, 5))
for layer, group in weight_variance_df.groupby("layer"):
    group = group.sort_values("N")
    ax.plot(group["N"], group["variance"], marker="o", label=layer)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Width N")
ax.set_ylabel("Trained weight variance")
ax.set_title("Weight variance scaling with width")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS_DIR / f"weight_variance_vs_width_layer_{SUMMARY_LAYER_IDX}.png", dpi=150)
plt.show()


## Activation Variance vs Width

For one saved test sample, plot the variance across coordinates of each activation vector after the embedding, after each residual block, and after the readout as a function of width.


In [ ]:
ACTIVATION_SAMPLE_IDX = 2
ACTIVATION_CHECKPOINT_STEP = "final"  # Use "final" or an integer step.

from parity_net.checkpoint import load_checkpoint
from parity_net.data import load_dataset
from parity_net.train import resolve_device, resolve_dtype


def activation_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('ACTIVATION_CHECKPOINT_STEP must be "final" or an integer step')


activation_checkpoint_file, activation_checkpoint_label = activation_checkpoint_name(ACTIVATION_CHECKPOINT_STEP)
activation_variance_rows = []
for N in WIDTHS:
    run_dir = RUNS_DIR / f"N_{N}"
    checkpoint_path = run_dir / "checkpoints" / activation_checkpoint_file
    if not checkpoint_path.exists():
        print(f"N={N}: missing checkpoint {checkpoint_path}, skipping activation variance")
        continue

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, payload, _ = load_checkpoint(checkpoint_path, device)
    training = payload["config"]["training"]
    device = resolve_device(training["device"])
    dtype = resolve_dtype(training["dtype"])
    model = model.to(device=device, dtype=dtype)

    test_data_path = Path(payload.get("test_data_path") or run_dir / "test_data.pt")
    test_data = load_dataset(test_data_path, device, dtype)
    if ACTIVATION_SAMPLE_IDX < 0 or ACTIVATION_SAMPLE_IDX >= test_data.x.shape[0]:
        raise ValueError(
            f"ACTIVATION_SAMPLE_IDX={ACTIVATION_SAMPLE_IDX} is outside saved test set "
            f"of size {test_data.x.shape[0]} for N={N}"
        )

    x_sample = test_data.x[ACTIVATION_SAMPLE_IDX : ACTIVATION_SAMPLE_IDX + 1]
    model.eval()
    with torch.no_grad():
        readout, activations = model(x_sample, return_activations=True)

    for layer_idx, h in enumerate(activations):
        layer_name = "embedding" if layer_idx == 0 else f"block_{layer_idx}"
        activation_variance_rows.append(
            {
                "N": N,
                "checkpoint": activation_checkpoint_label,
                "layer_idx": layer_idx,
                "layer": layer_name,
                "variance": h.squeeze(0).detach().float().var(unbiased=False).item(),
            }
        )

    activation_variance_rows.append(
        {
            "N": N,
            "checkpoint": activation_checkpoint_label,
            "layer_idx": len(activations),
            "layer": "readout",
            "variance": readout.squeeze(0).detach().float().var(unbiased=False).item(),
        }
    )

activation_variance_by_width_df = pd.DataFrame(activation_variance_rows)
activation_variance_by_width_df.to_csv(
    ANALYSIS_DIR / f"activation_variance_by_width_{activation_checkpoint_label}.csv",
    index=False,
)
display(activation_variance_by_width_df)

fig, ax = plt.subplots(figsize=(8, 5))
for layer, group in activation_variance_by_width_df.groupby("layer"):
    group = group.sort_values("N")
    ax.plot(group["N"], group["variance"], marker="o", label=layer)
ax.set_xscale("log", base=2)
ax.set_xlabel("Width N")
ax.set_ylabel("Activation-vector variance")
ax.set_title(
    f"Activation variance vs width, {activation_checkpoint_label}, "
    f"test sample {ACTIVATION_SAMPLE_IDX}"
)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / f"activation_variance_by_width_{activation_checkpoint_label}.png", dpi=150)
plt.show()


In [ ]:
run_dir_ = run_dir.parent / "N_2048"

ckpt_a = torch.load(run_dir_/ "checkpoints" / "step_00001000.pt", map_location="cpu")
ckpt_b = torch.load(run_dir_/ "checkpoints" / "final.pt", map_location="cpu")

w_a = ckpt_a["model_state"]["embedding.weight"]
w_b = ckpt_b["model_state"]["embedding.weight"]

print("embedding delta norm:", (w_b - w_a).norm().item())
print("relative delta:", (w_b - w_a).norm().item() / w_a.norm().item())


## Neuron-Target Correlations

Load a checkpoint from the run directory for a chosen width `N`, compute Pearson correlations between each neuron activation and each saved-test-set target, then inspect any selected neuron.


In [ ]:
import pandas as pd
import torch

from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset, target_names
from parity_net.train import resolve_device, resolve_dtype

CORRELATION_N = 2048
CORRELATION_CHECKPOINT_STEP = "final"  # Use "final" or an integer step, e.g. 1000.
CORRELATION_SAMPLES = 20_000
CORRELATION_ACTIVATION_SOURCE = "block_nonlinearity"  # "residual_stream" or "block_nonlinearity".

def correlation_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('CORRELATION_CHECKPOINT_STEP must be "final" or an integer step')


def collect_correlation_activations(model, x, source):
    if source == "residual_stream":
        _, activations = model(x, return_activations=True)
        layer_names = ["embedding", *[f"after_block_{i}" for i in range(len(model.blocks))]]
        return activations, layer_names

    if source == "block_nonlinearity":
        h = model.embedding(x)
        activations = []
        layer_names = []
        for block_idx, block in enumerate(model.blocks):
            nonlinear = block.activation(block.linear(h))
            activations.append(nonlinear)
            layer_names.append(f"block_{block_idx}_nonlinearity")
            update = nonlinear
            if block.post_activation_linear is not None:
                update = block.post_activation_linear(update)
            h = h + update
        return activations, layer_names

    raise ValueError(
        'CORRELATION_ACTIVATION_SOURCE must be "residual_stream" or "block_nonlinearity"'
    )


correlation_checkpoint_file, correlation_checkpoint_label = correlation_checkpoint_name(
    CORRELATION_CHECKPOINT_STEP
)
correlation_run_dir = RUNS_DIR / f"N_{CORRELATION_N}"
correlation_checkpoint_path = correlation_run_dir / "checkpoints" / correlation_checkpoint_file
if not correlation_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {correlation_checkpoint_path}")

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, payload, _ = load_checkpoint(correlation_checkpoint_path, load_device)
training = payload["config"]["training"]
model_config = payload["config"]["model"]
task_config = payload["config"].get("task") or {
    "input_dim": model_config["input_dim"],
    "relevant_dim": model_config["relevant_dim"],
    "exclude_targets": [],
}
target_columns = target_names(
    int(task_config["relevant_dim"]),
    list(task_config.get("exclude_targets", [])),
)
device = resolve_device(training["device"])
dtype = resolve_dtype(training["dtype"])
model = model.to(device=device, dtype=dtype)
model.eval()

test_data_path = Path(payload.get("test_data_path") or correlation_run_dir / "test_data.pt")
if not test_data_path.exists():
    raise FileNotFoundError(f"Missing saved test data: {test_data_path}")
test_data = load_dataset(test_data_path, device, dtype)
correlation_count = min(CORRELATION_SAMPLES, test_data.x.shape[0])
correlation_data = ParityDataset(
    x=test_data.x[:correlation_count],
    y=test_data.y[:correlation_count],
)

with torch.no_grad():
    correlation_activations, correlation_layer_names = collect_correlation_activations(
        model,
        correlation_data.x,
        CORRELATION_ACTIVATION_SOURCE,
    )

if not correlation_activations:
    raise ValueError(
        f"No activations were collected for source {CORRELATION_ACTIVATION_SOURCE!r}. "
        "This can happen if the model has no residual blocks."
    )

if correlation_data.y.shape[1] != len(target_columns):
    raise ValueError(
        f"Expected {len(target_columns)} target labels, got {correlation_data.y.shape[1]} targets"
    )

y = correlation_data.y.detach().float()
y_centered = y - y.mean(dim=0, keepdim=True)
y_norm = torch.linalg.vector_norm(y_centered, dim=0).clamp_min(torch.finfo(y_centered.dtype).eps)

correlation_by_layer = {}
correlation_rows = []
for layer_idx, (h, layer_name) in enumerate(zip(correlation_activations, correlation_layer_names)):
    h = h.detach().float()
    if not torch.isfinite(h).all().item():
        raise ValueError(f"Layer {layer_idx} ({layer_name}) activations contain NaN or Inf")
    h_centered = h - h.mean(dim=0, keepdim=True)
    h_norm = torch.linalg.vector_norm(h_centered, dim=0).clamp_min(torch.finfo(h_centered.dtype).eps)
    corr = (h_centered.T @ y_centered) / (h_norm[:, None] * y_norm[None, :])
    corr_cpu = corr.cpu()
    correlation_by_layer[layer_idx] = corr_cpu
    for neuron_idx in range(corr_cpu.shape[0]):
        row = {
            "N": CORRELATION_N,
            "checkpoint": correlation_checkpoint_label,
            "activation_source": CORRELATION_ACTIVATION_SOURCE,
            "layer_idx": layer_idx,
            "layer": layer_name,
            "neuron_idx": neuron_idx,
            "max_abs_correlation": corr_cpu[neuron_idx].abs().max().item(),
        }
        row.update(
            {
                target: corr_cpu[neuron_idx, target_idx].item()
                for target_idx, target in enumerate(target_columns)
            }
        )
        correlation_rows.append(row)

neuron_target_correlations = pd.DataFrame(correlation_rows)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
correlation_csv_path = (
    ANALYSIS_DIR
    / f"neuron_target_correlations_N_{CORRELATION_N}_{correlation_checkpoint_label}_{CORRELATION_ACTIVATION_SOURCE}.csv"
)
neuron_target_correlations.to_csv(correlation_csv_path, index=False)

print(
    f"Computed {CORRELATION_ACTIVATION_SOURCE} correlations for N={CORRELATION_N}, "
    f"checkpoint={correlation_checkpoint_label}, "
    f"{len(correlation_by_layer)} activation layers, "
    f"{correlation_by_layer[0].shape[0]} neurons per collected layer, "
    f"and {len(target_columns)} targets using {correlation_count} saved test samples."
)
print("Layer index map:")
for layer_idx, layer_name in enumerate(correlation_layer_names):
    print(f"  {layer_idx}: {layer_name}")
print(f"Saved correlations to {correlation_csv_path}")
display(neuron_target_correlations.head())
display(
    neuron_target_correlations
    .sort_values("max_abs_correlation", ascending=False)
    .head(20)
)


In [ ]:
import matplotlib.pyplot as plt

SELECT_LAYER_IDX = 0
SELECT_NEURON_IDX = 0

if SELECT_LAYER_IDX not in correlation_by_layer:
    raise ValueError(f"SELECT_LAYER_IDX must be one of {list(correlation_by_layer)}")
selected_corr = correlation_by_layer[SELECT_LAYER_IDX]
if SELECT_NEURON_IDX < 0 or SELECT_NEURON_IDX >= selected_corr.shape[0]:
    raise ValueError(f"SELECT_NEURON_IDX must be in [0, {selected_corr.shape[0] - 1}]")

selected_neuron_correlations = pd.DataFrame(
    {
        "target": target_columns,
        "correlation": selected_corr[SELECT_NEURON_IDX].numpy(),
    }
)
selected_neuron_correlations["abs_correlation"] = selected_neuron_correlations["correlation"].abs()
selected_neuron_correlations = selected_neuron_correlations.sort_values(
    "abs_correlation",
    ascending=False,
)

layer_name = correlation_layer_names[SELECT_LAYER_IDX]
print(
    f"Correlations for N={CORRELATION_N}, {correlation_checkpoint_label}, "
    f"{CORRELATION_ACTIVATION_SOURCE}, {layer_name}, neuron {SELECT_NEURON_IDX}"
)
display(selected_neuron_correlations)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 4))
selected_neuron_correlations.sort_values("target").plot(
    x="target",
    y="correlation",
    kind="bar",
    legend=False,
    ax=ax,
)
ax.axhline(0.0, color="black", linewidth=1)
ax.set_xlabel("Target")
ax.set_ylabel("Pearson correlation")
ax.set_title(
    f"N={CORRELATION_N}, {correlation_checkpoint_label}, {CORRELATION_ACTIVATION_SOURCE}, "
    f"{layer_name} neuron {SELECT_NEURON_IDX}"
)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plot_path = (
    PLOTS_DIR
    / f"neuron_target_correlations_N_{CORRELATION_N}_{correlation_checkpoint_label}_{CORRELATION_ACTIVATION_SOURCE}_layer_{SELECT_LAYER_IDX}_neuron_{SELECT_NEURON_IDX}.png"
)
fig.savefig(plot_path, dpi=150)
print(f"Saved plot to {plot_path}")
plt.show()


## Thresholded Neuron-Target Correlation Counts

Choose a layer and threshold, then count neurons that are correlated with at least one output parity above that threshold.


In [ ]:
CORRELATION_LAYER_IDX_FOR_COUNTS = 0
CORRELATION_THRESHOLD_FOR_COUNTS = 0.1
USE_ABSOLUTE_CORRELATION_FOR_COUNTS = True

if not 0.0 <= CORRELATION_THRESHOLD_FOR_COUNTS <= 1.0:
    raise ValueError("CORRELATION_THRESHOLD_FOR_COUNTS must be between 0 and 1")
if CORRELATION_LAYER_IDX_FOR_COUNTS not in correlation_by_layer:
    raise ValueError(f"Layer must be one of {list(correlation_by_layer)}")

count_layer_corr = correlation_by_layer[CORRELATION_LAYER_IDX_FOR_COUNTS]
count_layer_name = correlation_layer_names[CORRELATION_LAYER_IDX_FOR_COUNTS]
count_layer_scores = count_layer_corr.abs() if USE_ABSOLUTE_CORRELATION_FOR_COUNTS else count_layer_corr
count_neuron_mask = (count_layer_scores > CORRELATION_THRESHOLD_FOR_COUNTS).any(dim=1)
count_neuron_indices = torch.nonzero(count_neuron_mask, as_tuple=False).flatten().tolist()
num_count_neurons = len(count_neuron_indices)
total_count_neurons = count_layer_corr.shape[0]

print(
    f"Layer {CORRELATION_LAYER_IDX_FOR_COUNTS} ({count_layer_name}), "
    f"source={CORRELATION_ACTIVATION_SOURCE}: "
    f"{num_count_neurons}/{total_count_neurons} neurons have at least one "
    f"{'absolute ' if USE_ABSOLUTE_CORRELATION_FOR_COUNTS else ''}"
    f"correlation > {CORRELATION_THRESHOLD_FOR_COUNTS}."
)


## Monosemanticity Table

For the selected layer, list neurons whose correlation with at least one output parity exceeds the threshold, together with the matching parity names and coefficients.


In [ ]:
MONOSEMANTICITY_LAYER_IDX = 0
MONOSEMANTICITY_THRESHOLD = 0.1
MONOSEMANTICITY_USE_ABSOLUTE_CORRELATION = True

if not 0.0 <= MONOSEMANTICITY_THRESHOLD <= 1.0:
    raise ValueError("MONOSEMANTICITY_THRESHOLD must be between 0 and 1")
if MONOSEMANTICITY_LAYER_IDX not in correlation_by_layer:
    raise ValueError(f"Layer must be one of {list(correlation_by_layer)}")


def parity_degree(target_name):
    return target_name.split("_", 1)[0]


def semanticity_category(correlated_targets):
    if len(correlated_targets) == 1:
        return "monosemantic"
    degrees = {parity_degree(target) for target in correlated_targets}
    if len(degrees) > 1:
        return "cross_degree_polysemantic"
    if len(correlated_targets) == 2:
        return "pure_degree_polysemantic_two_parities"
    return "pure_degree_polysemantic_more_than_two_parities"


monosemanticity_layer_corr = correlation_by_layer[MONOSEMANTICITY_LAYER_IDX]
monosemanticity_layer_name = correlation_layer_names[MONOSEMANTICITY_LAYER_IDX]
monosemanticity_layer_scores = (
    monosemanticity_layer_corr.abs()
    if MONOSEMANTICITY_USE_ABSOLUTE_CORRELATION
    else monosemanticity_layer_corr
)
monosemanticity_neuron_mask = (monosemanticity_layer_scores > MONOSEMANTICITY_THRESHOLD).any(dim=1)
monosemanticity_neuron_indices = torch.nonzero(
    monosemanticity_neuron_mask,
    as_tuple=False,
).flatten().tolist()

total_layer_neurons = monosemanticity_layer_corr.shape[0]
monosemanticity_rows = []
for neuron_idx in monosemanticity_neuron_indices:
    corr_values = monosemanticity_layer_corr[neuron_idx]
    score_values = monosemanticity_layer_scores[neuron_idx]
    hit_indices = torch.nonzero(score_values > MONOSEMANTICITY_THRESHOLD, as_tuple=False).flatten().tolist()
    hit_indices = sorted(hit_indices, key=lambda target_idx: score_values[target_idx].item(), reverse=True)
    correlated_targets = [target_columns[target_idx] for target_idx in hit_indices]
    correlated_degrees = [parity_degree(target) for target in correlated_targets]
    correlated_coefficients = [corr_values[target_idx].item() for target_idx in hit_indices]
    monosemanticity_rows.append(
        {
            "layer_idx": MONOSEMANTICITY_LAYER_IDX,
            "layer": monosemanticity_layer_name,
            "activation_source": CORRELATION_ACTIVATION_SOURCE,
            "neuron_idx": neuron_idx,
            "num_correlated_outputs": len(hit_indices),
            "num_correlated_degrees": len(set(correlated_degrees)),
            "semanticity_category": semanticity_category(correlated_targets),
            "max_abs_correlation": corr_values.abs().max().item(),
            "correlated_outputs": ", ".join(correlated_targets),
            "correlated_degrees": ", ".join(correlated_degrees),
            "correlation_coefficients": ", ".join(
                f"{target}: {coefficient:.4f}"
                for target, coefficient in zip(correlated_targets, correlated_coefficients)
            ),
        }
    )

monosemanticity_df = pd.DataFrame(monosemanticity_rows)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
threshold_label = str(MONOSEMANTICITY_THRESHOLD).replace(".", "p")
monosemanticity_csv_path = (
    ANALYSIS_DIR
    / f"monosemanticity_N_{CORRELATION_N}_{correlation_checkpoint_label}_{CORRELATION_ACTIVATION_SOURCE}_layer_{MONOSEMANTICITY_LAYER_IDX}_threshold_{threshold_label}.csv"
)
monosemanticity_summary_csv_path = (
    ANALYSIS_DIR
    / f"monosemanticity_summary_N_{CORRELATION_N}_{correlation_checkpoint_label}_{CORRELATION_ACTIVATION_SOURCE}_layer_{MONOSEMANTICITY_LAYER_IDX}_threshold_{threshold_label}.csv"
)

if monosemanticity_df.empty:
    monosemanticity_summary_df = pd.DataFrame(
        columns=[
            "semanticity_category",
            "num_neurons",
            "fraction_among_correlated_neurons",
            "fraction_among_all_neurons",
        ]
    )
    print(
        f"No neurons in layer {MONOSEMANTICITY_LAYER_IDX} ({monosemanticity_layer_name}) "
        f"passed threshold {MONOSEMANTICITY_THRESHOLD}."
    )
else:
    monosemanticity_df = monosemanticity_df.sort_values(
        ["num_correlated_outputs", "max_abs_correlation"],
        ascending=[True, False],
    )
    category_order = [
        "monosemantic",
        "pure_degree_polysemantic_two_parities",
        "cross_degree_polysemantic",
        "pure_degree_polysemantic_more_than_two_parities",
    ]
    category_counts = monosemanticity_df["semanticity_category"].value_counts()
    monosemanticity_summary_df = pd.DataFrame(
        {
            "semanticity_category": category_order,
            "num_neurons": [int(category_counts.get(category, 0)) for category in category_order],
        }
    )
    num_correlated_neurons = len(monosemanticity_df)
    monosemanticity_summary_df["fraction_among_correlated_neurons"] = (
        monosemanticity_summary_df["num_neurons"] / num_correlated_neurons
    )
    monosemanticity_summary_df["fraction_among_all_neurons"] = (
        monosemanticity_summary_df["num_neurons"] / total_layer_neurons
    )
    print(
        f"Layer {MONOSEMANTICITY_LAYER_IDX} ({monosemanticity_layer_name}), "
        f"source={CORRELATION_ACTIVATION_SOURCE}: "
        f"{len(monosemanticity_df)}/{total_layer_neurons} neurons passed threshold {MONOSEMANTICITY_THRESHOLD}."
    )
    print("Semanticity fractions:")
    display(monosemanticity_summary_df)
    display(monosemanticity_df)

monosemanticity_df.to_csv(monosemanticity_csv_path, index=False)
monosemanticity_summary_df.to_csv(monosemanticity_summary_csv_path, index=False)
print(f"Saved monosemanticity table to {monosemanticity_csv_path}")
print(f"Saved monosemanticity summary to {monosemanticity_summary_csv_path}")


## Proportion Correlated With One Output

For a chosen layer and output parity name, compute the proportion of neurons whose correlation with that output exceeds the threshold.


In [ ]:
PROPORTION_LAYER_IDX = 0
OUTPUT_PARITY = "d2_0"
PROPORTION_THRESHOLD = 0.5
PROPORTION_USE_ABSOLUTE_CORRELATION = True

if not 0.0 <= PROPORTION_THRESHOLD <= 1.0:
    raise ValueError("PROPORTION_THRESHOLD must be between 0 and 1")
if PROPORTION_LAYER_IDX not in correlation_by_layer:
    raise ValueError(f"Layer must be one of {list(correlation_by_layer)}")
if OUTPUT_PARITY not in target_columns:
    raise ValueError(f"OUTPUT_PARITY must be one of {target_columns}")

proportion_corr = correlation_by_layer[PROPORTION_LAYER_IDX]
proportion_layer_name = correlation_layer_names[PROPORTION_LAYER_IDX]
target_idx = target_columns.index(OUTPUT_PARITY)
output_corr = proportion_corr[:, target_idx]
output_scores = output_corr.abs() if PROPORTION_USE_ABSOLUTE_CORRELATION else output_corr
output_neuron_mask = output_scores > PROPORTION_THRESHOLD
output_neuron_indices = torch.nonzero(output_neuron_mask, as_tuple=False).flatten().tolist()
output_num_neurons = len(output_neuron_indices)
output_total_neurons = output_corr.shape[0]
output_proportion = output_num_neurons / output_total_neurons

print(
    f"Layer {PROPORTION_LAYER_IDX} ({proportion_layer_name}), source={CORRELATION_ACTIVATION_SOURCE}, "
    f"output={OUTPUT_PARITY}: {output_num_neurons}/{output_total_neurons} neurons "
    f"({output_proportion:.4f}) have "
    f"{'absolute ' if PROPORTION_USE_ABSOLUTE_CORRELATION else ''}correlation > {PROPORTION_THRESHOLD}."
)

output_proportion_df = pd.DataFrame(
    {
        "layer_idx": [PROPORTION_LAYER_IDX],
        "layer": [proportion_layer_name],
        "activation_source": [CORRELATION_ACTIVATION_SOURCE],
        "output_parity": [OUTPUT_PARITY],
        "threshold": [PROPORTION_THRESHOLD],
        "use_absolute_correlation": [PROPORTION_USE_ABSOLUTE_CORRELATION],
        "num_neurons": [output_num_neurons],
        "total_neurons": [output_total_neurons],
        "proportion": [output_proportion],
    }
)
display(output_proportion_df)


## Embedding Matrix Norms and Cosines

Load a checkpoint, inspect the `N x input_dim` embedding matrix, display the norm of each input-coordinate embedding vector, and show cosine similarities among the first 16 embedding vectors.


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

from parity_net.checkpoint import load_checkpoint
from parity_net.train import resolve_device, resolve_dtype

EMBEDDING_MATRIX_N = 2048
EMBEDDING_MATRIX_CHECKPOINT_STEP = "final"  # Use "final" or an integer step, e.g. 1000.
NUM_EMBEDDINGS_FOR_COSINE = 16


def embedding_matrix_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('EMBEDDING_MATRIX_CHECKPOINT_STEP must be "final" or an integer step')


embedding_matrix_checkpoint_file, embedding_matrix_checkpoint_label = embedding_matrix_checkpoint_name(
    EMBEDDING_MATRIX_CHECKPOINT_STEP
)
embedding_matrix_run_dir = RUNS_DIR / f"N_{EMBEDDING_MATRIX_N}"
embedding_matrix_checkpoint_path = embedding_matrix_run_dir / "checkpoints" / embedding_matrix_checkpoint_file
if not embedding_matrix_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {embedding_matrix_checkpoint_path}")

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model, embedding_payload, _ = load_checkpoint(embedding_matrix_checkpoint_path, load_device)
training = embedding_payload["config"]["training"]
device = resolve_device(training["device"])
dtype = resolve_dtype(training["dtype"])
embedding_model = embedding_model.to(device=device, dtype=dtype)
embedding_model.eval()

embedding_matrix = embedding_model.embedding.weight.detach().float()
if embedding_matrix.ndim != 2:
    raise ValueError(f"Expected a 2D embedding matrix, got shape {tuple(embedding_matrix.shape)}")

input_embeddings = embedding_matrix.T
embedding_norms = torch.linalg.vector_norm(input_embeddings, dim=1)
embedding_norms_df = pd.DataFrame(
    {
        "input_index": list(range(input_embeddings.shape[0])),
        "embedding_norm": embedding_norms.cpu().numpy(),
    }
)

num_cosine = min(NUM_EMBEDDINGS_FOR_COSINE, input_embeddings.shape[0])
first_embeddings = input_embeddings[:num_cosine]
first_embeddings_normalized = F.normalize(first_embeddings, p=2, dim=1)
embedding_cosine_matrix = first_embeddings_normalized @ first_embeddings_normalized.T
embedding_cosine_df = pd.DataFrame(
    embedding_cosine_matrix.cpu().numpy(),
    index=[f"input_{idx}" for idx in range(num_cosine)],
    columns=[f"input_{idx}" for idx in range(num_cosine)],
)

print(
    f"Loaded N={EMBEDDING_MATRIX_N}, checkpoint={embedding_matrix_checkpoint_label}. "
    f"Embedding matrix shape: {tuple(embedding_matrix.shape)}"
)
print("Norms of input-coordinate embedding vectors:")
display(embedding_norms_df)
print(f"Cosine similarity matrix for the first {num_cosine} input-coordinate embedding vectors:")
display(embedding_cosine_df)


## Readout Matrix SVD

Load a checkpoint, extract the `15 x N` readout matrix, compute its SVD, and print the singular values.


In [ ]:
import pandas as pd
import torch

from parity_net.checkpoint import load_checkpoint
from parity_net.data import target_names
from parity_net.train import resolve_device, resolve_dtype

READOUT_SVD_N = 2048
READOUT_SVD_CHECKPOINT_STEP = "final"  # Use "final" or an integer step, e.g. 1000.


def readout_svd_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('READOUT_SVD_CHECKPOINT_STEP must be "final" or an integer step')


readout_svd_checkpoint_file, readout_svd_checkpoint_label = readout_svd_checkpoint_name(
    READOUT_SVD_CHECKPOINT_STEP
)
readout_svd_run_dir = RUNS_DIR / f"N_{READOUT_SVD_N}"
readout_svd_checkpoint_path = readout_svd_run_dir / "checkpoints" / readout_svd_checkpoint_file
if not readout_svd_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {readout_svd_checkpoint_path}")

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
readout_svd_model, readout_svd_payload, _ = load_checkpoint(readout_svd_checkpoint_path, load_device)
training = readout_svd_payload["config"]["training"]
device = resolve_device(training["device"])
dtype = resolve_dtype(training["dtype"])
readout_svd_model = readout_svd_model.to(device=device, dtype=dtype)
readout_svd_model.eval()

readout_matrix = readout_svd_model.readout_weight_matrix().detach().float()
if readout_matrix.ndim != 2:
    raise ValueError(f"Expected a 2D readout matrix, got shape {tuple(readout_matrix.shape)}")
if not torch.isfinite(readout_matrix).all().item():
    raise ValueError("Readout matrix contains NaN or Inf")

readout_u, readout_singular_values, readout_vh = torch.linalg.svd(readout_matrix, full_matrices=False)
readout_singular_values_df = pd.DataFrame(
    {
        "index": list(range(readout_singular_values.numel())),
        "singular_value": readout_singular_values.detach().cpu().numpy(),
    }
)
readout_model_config = readout_svd_payload["config"]["model"]
readout_task_config = readout_svd_payload["config"].get("task") or {
    "input_dim": readout_model_config["input_dim"],
    "relevant_dim": readout_model_config["relevant_dim"],
    "exclude_targets": [],
}
readout_target_names = target_names(
    int(readout_task_config["relevant_dim"]),
    list(readout_task_config.get("exclude_targets", [])),
)
left_singular_vector_columns = [f"left_sv_{idx}" for idx in range(readout_u.shape[1])]
readout_left_singular_vectors_df = pd.DataFrame(
    readout_u.detach().cpu().numpy(),
    index=readout_target_names,
    columns=left_singular_vector_columns,
)

print(
    f"Loaded N={READOUT_SVD_N}, checkpoint={readout_svd_checkpoint_label}. "
    f"Readout matrix shape: {tuple(readout_matrix.shape)}"
)
print("Singular values:")
print(readout_singular_values.detach().cpu().numpy())
display(readout_singular_values_df)
print("Left singular vectors (rows are output targets, columns are left singular-vector indices):")
print(readout_u.detach().cpu().numpy())
display(readout_left_singular_vectors_df)


Compute cosine similarities among readout right singular vectors, and between input-coordinate embedding vectors and those singular vectors.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

if "readout_svd_model" not in globals() or "readout_vh" not in globals():
    raise NameError("Run the Readout Matrix SVD cell first")

right_singular_vectors = readout_vh.detach().float()
right_singular_vectors = F.normalize(right_singular_vectors, dim=1, eps=1e-12)
right_singular_cosine = right_singular_vectors @ right_singular_vectors.T

num_singular_vectors = right_singular_vectors.shape[0]
sv_labels = [f"sv_{i}" for i in range(num_singular_vectors)]
right_singular_cosine_df = pd.DataFrame(
    right_singular_cosine.detach().cpu().numpy(),
    index=sv_labels,
    columns=sv_labels,
)

embedding_weight = readout_svd_model.embedding.weight.detach().float()
if embedding_weight.ndim != 2:
    raise ValueError(f"Expected a 2D embedding matrix, got shape {tuple(embedding_weight.shape)}")
if embedding_weight.shape[0] != right_singular_vectors.shape[1]:
    raise ValueError(
        "Embedding vectors and right singular vectors live in different dimensions: "
        f"embedding width={embedding_weight.shape[0]}, singular-vector width={right_singular_vectors.shape[1]}"
    )
if not torch.isfinite(embedding_weight).all().item():
    raise ValueError("Embedding matrix contains NaN or Inf")

input_dim = embedding_weight.shape[1]
embedding_vectors = embedding_weight.T[:input_dim]
embedding_vectors = F.normalize(embedding_vectors, dim=1, eps=1e-12)
embedding_right_singular_cosine = embedding_vectors @ right_singular_vectors.T

embedding_labels = [f"input_{i}" for i in range(input_dim)]
embedding_right_singular_cosine_df = pd.DataFrame(
    embedding_right_singular_cosine.detach().cpu().numpy(),
    index=embedding_labels,
    columns=sv_labels,
)

right_singular_cosine_path = (
    ANALYSIS_DIR
    / f"readout_right_singular_vector_cosines_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}.csv"
)
embedding_right_singular_cosine_path = (
    ANALYSIS_DIR
    / f"embedding_readout_right_singular_cosines_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}.csv"
)
right_singular_cosine_df.to_csv(right_singular_cosine_path)
embedding_right_singular_cosine_df.to_csv(embedding_right_singular_cosine_path)

print(f"Right singular vectors shape: {tuple(right_singular_vectors.shape)}")
print(f"Embedding vectors shape: {tuple(embedding_vectors.shape)}")
print(f"Saved right singular vector cosine matrix to {right_singular_cosine_path}")
print(f"Saved embedding/right-singular cosine matrix to {embedding_right_singular_cosine_path}")

display(right_singular_cosine_df)
display(embedding_right_singular_cosine_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
image0 = axes[0].imshow(right_singular_cosine_df.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_title("Right singular vector cosine similarities")
axes[0].set_xlabel("Right singular vector")
axes[0].set_ylabel("Right singular vector")
axes[0].set_xticks(range(num_singular_vectors), sv_labels, rotation=90)
axes[0].set_yticks(range(num_singular_vectors), sv_labels)
fig.colorbar(image0, ax=axes[0], fraction=0.046, pad=0.04)

image1 = axes[1].imshow(embedding_right_singular_cosine_df.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
axes[1].set_title("Embedding vectors vs right singular vectors")
axes[1].set_xlabel("Right singular vector")
axes[1].set_ylabel("Input-coordinate embedding")
axes[1].set_xticks(range(num_singular_vectors), sv_labels, rotation=90)
axes[1].set_yticks(range(input_dim), embedding_labels)
fig.colorbar(image1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
plot_path = PLOTS_DIR / f"readout_svd_embedding_cosines_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved cosine similarity plot to {plot_path}")


Project one residual-block activation onto input embedding directions and readout right singular vectors.


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

if "readout_svd_model" not in globals() or "readout_vh" not in globals():
    raise NameError("Run the Readout Matrix SVD cell first")

SVD_PROJECTION_BLOCK_IDX = 0  # 0 means after the first residual block.
SVD_PROJECTION_RELEVANT_BITS = [1] * 16  # Replace with a list like [1, -1, ..., 1]. If None, uses all +1.
SVD_PROJECTION_RELEVANT_BITS[0] = 1   # Replace with a list like [1, -1, ..., 1]. If None, uses all +1.
SVD_PROJECTION_RANDOM_TAIL_SEED = 0

svd_projection_config = readout_svd_payload["config"]
svd_projection_model_config = svd_projection_config["model"]
svd_projection_task_config = svd_projection_config.get("task") or {
    "input_dim": svd_projection_model_config["input_dim"],
    "relevant_dim": svd_projection_model_config["relevant_dim"],
    "exclude_targets": [],
}
input_dim = int(svd_projection_task_config["input_dim"])
relevant_dim = int(svd_projection_task_config["relevant_dim"])

num_blocks = len(readout_svd_model.blocks)
if SVD_PROJECTION_BLOCK_IDX < 0 or SVD_PROJECTION_BLOCK_IDX >= num_blocks:
    raise ValueError(f"SVD_PROJECTION_BLOCK_IDX must be in [0, {num_blocks - 1}]")

if SVD_PROJECTION_RELEVANT_BITS is None:
    SVD_PROJECTION_RELEVANT_BITS = [1] * relevant_dim
    print(
        "SVD_PROJECTION_RELEVANT_BITS was None, so using all +1 relevant bits. "
        "Edit SVD_PROJECTION_RELEVANT_BITS to provide a specific relevant sequence."
    )
if len(SVD_PROJECTION_RELEVANT_BITS) != relevant_dim:
    raise ValueError(
        f"Expected {relevant_dim} relevant bits, got {len(SVD_PROJECTION_RELEVANT_BITS)}"
    )
if any(bit not in {-1, 1} for bit in SVD_PROJECTION_RELEVANT_BITS):
    raise ValueError("SVD_PROJECTION_RELEVANT_BITS must contain only -1 and 1")

load_device = next(readout_svd_model.parameters()).device
dtype = next(readout_svd_model.parameters()).dtype
relevant_bits = torch.tensor(SVD_PROJECTION_RELEVANT_BITS, device=load_device, dtype=dtype)

generator = torch.Generator(device="cpu")
generator.manual_seed(SVD_PROJECTION_RANDOM_TAIL_SEED)
tail_dim = input_dim - relevant_dim
if tail_dim > 0:
    random_tail = torch.randint(0, 2, (tail_dim,), generator=generator).to(device=load_device, dtype=dtype)
    random_tail = random_tail.mul(2).sub(1)
    input_sequence = torch.cat([relevant_bits, random_tail], dim=0)
else:
    input_sequence = relevant_bits
x = input_sequence.unsqueeze(0)

readout_svd_model.eval()
with torch.no_grad():
    output, activations = readout_svd_model(x, return_activations=True)
activation = activations[SVD_PROJECTION_BLOCK_IDX + 1].squeeze(0).detach().float().cpu()
output = output.squeeze(0).detach().float().cpu()

embedding_weight = readout_svd_model.embedding.weight.detach().float().cpu()
embedding_directions = embedding_weight[:, :relevant_dim].T
embedding_directions = F.normalize(embedding_directions, dim=1, eps=1e-12)
embedding_projection_values = embedding_directions @ activation

right_singular_directions = F.normalize(readout_vh.detach().float().cpu(), dim=1, eps=1e-12)
right_singular_projection_values = right_singular_directions @ activation

input_sequence_df = pd.DataFrame(
    {
        "position": list(range(input_dim)),
        "value": input_sequence.detach().cpu().to(torch.int64).numpy(),
        "is_relevant": [idx < relevant_dim for idx in range(input_dim)],
    }
)
embedding_projection_df = pd.DataFrame(
    {
        "input_position": list(range(relevant_dim)),
        "projection_on_embedding_direction": embedding_projection_values.numpy(),
    }
)
right_singular_projection_df = pd.DataFrame(
    {
        "singular_vector_idx": list(range(right_singular_projection_values.numel())),
        "singular_value": readout_singular_values.detach().float().cpu().numpy(),
        "projection_on_right_singular_vector": right_singular_projection_values.numpy(),
    }
)
output_df = pd.DataFrame(
    {
        "output_idx": list(range(output.numel())),
        "model_output": output.numpy(),
    }
)

prefix = f"activation_projection_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}_block_{SVD_PROJECTION_BLOCK_IDX}"
input_sequence_path = ANALYSIS_DIR / f"{prefix}_input_sequence.csv"
embedding_projection_path = ANALYSIS_DIR / f"{prefix}_embedding_directions.csv"
right_singular_projection_path = ANALYSIS_DIR / f"{prefix}_right_singular_vectors.csv"
output_path = ANALYSIS_DIR / f"{prefix}_model_output.csv"
input_sequence_df.to_csv(input_sequence_path, index=False)
embedding_projection_df.to_csv(embedding_projection_path, index=False)
right_singular_projection_df.to_csv(right_singular_projection_path, index=False)
output_df.to_csv(output_path, index=False)

print(
    f"Captured activation after residual block {SVD_PROJECTION_BLOCK_IDX} "
    f"with shape {tuple(activation.shape)}."
)
print(f"Saved projection tables with prefix {prefix}")
print("Input sequence:")
display(input_sequence_df)
print("Model output:")
display(output_df)
print("Projection onto first relevant embedding directions:")
display(embedding_projection_df)
print("Projection onto readout right singular vectors:")
display(right_singular_projection_df)


Project one residual block's nonlinearity output, before residual addition, onto input embedding directions and readout right singular vectors.


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

if "readout_svd_model" not in globals() or "readout_vh" not in globals():
    raise NameError("Run the Readout Matrix SVD cell first")

NONLINEARITY_PROJECTION_BLOCK_IDX = 0  # 0 means the first residual block.
NONLINEARITY_PROJECTION_RELEVANT_BITS = [1] * 16  # Replace with a list like [1, -1, ..., 1]. If None, uses all +1.
# NONLINEARITY_PROJECTION_RELEVANT_BITS[0] = -1
# NONLINEARITY_PROJECTION_RELEVANT_BITS[1] = 1
NONLINEARITY_PROJECTION_RANDOM_TAIL_SEED = 0

nonlinearity_projection_config = readout_svd_payload["config"]
nonlinearity_projection_model_config = nonlinearity_projection_config["model"]
nonlinearity_projection_task_config = nonlinearity_projection_config.get("task") or {
    "input_dim": nonlinearity_projection_model_config["input_dim"],
    "relevant_dim": nonlinearity_projection_model_config["relevant_dim"],
    "exclude_targets": [],
}
input_dim = int(nonlinearity_projection_task_config["input_dim"])
relevant_dim = int(nonlinearity_projection_task_config["relevant_dim"])

num_blocks = len(readout_svd_model.blocks)
if NONLINEARITY_PROJECTION_BLOCK_IDX < 0 or NONLINEARITY_PROJECTION_BLOCK_IDX >= num_blocks:
    raise ValueError(f"NONLINEARITY_PROJECTION_BLOCK_IDX must be in [0, {num_blocks - 1}]")

if NONLINEARITY_PROJECTION_RELEVANT_BITS is None:
    NONLINEARITY_PROJECTION_RELEVANT_BITS = [1] * relevant_dim
    print(
        "NONLINEARITY_PROJECTION_RELEVANT_BITS was None, so using all +1 relevant bits. "
        "Edit NONLINEARITY_PROJECTION_RELEVANT_BITS to provide a specific relevant sequence."
    )
if len(NONLINEARITY_PROJECTION_RELEVANT_BITS) != relevant_dim:
    raise ValueError(
        f"Expected {relevant_dim} relevant bits, got {len(NONLINEARITY_PROJECTION_RELEVANT_BITS)}"
    )
if any(bit not in {-1, 1} for bit in NONLINEARITY_PROJECTION_RELEVANT_BITS):
    raise ValueError("NONLINEARITY_PROJECTION_RELEVANT_BITS must contain only -1 and 1")

load_device = next(readout_svd_model.parameters()).device
dtype = next(readout_svd_model.parameters()).dtype
relevant_bits = torch.tensor(NONLINEARITY_PROJECTION_RELEVANT_BITS, device=load_device, dtype=dtype)

generator = torch.Generator(device="cpu")
generator.manual_seed(NONLINEARITY_PROJECTION_RANDOM_TAIL_SEED)
tail_dim = input_dim - relevant_dim
if tail_dim > 0:
    random_tail = torch.randint(0, 2, (tail_dim,), generator=generator).to(device=load_device, dtype=dtype)
    random_tail = random_tail.mul(2).sub(1)
    input_sequence = torch.cat([relevant_bits, random_tail], dim=0)
else:
    input_sequence = relevant_bits
x = input_sequence.unsqueeze(0)

readout_svd_model.eval()
with torch.no_grad():
    full_output, _ = readout_svd_model(x, return_activations=True)
    h = readout_svd_model.embedding(x)
    for block_idx, block in enumerate(readout_svd_model.blocks):
        nonlinearity_output = block.activation(block.linear(h))
        if block_idx == NONLINEARITY_PROJECTION_BLOCK_IDX:
            captured_nonlinearity_output = nonlinearity_output.squeeze(0).detach().float().cpu()
            break
        if block.post_activation_linear is not None:
            residual_update = block.post_activation_linear(nonlinearity_output)
        else:
            residual_update = nonlinearity_output
        h = h + residual_update
    else:
        raise RuntimeError("Failed to capture the requested nonlinearity output")

full_output = full_output.squeeze(0).detach().float().cpu()
if not torch.isfinite(captured_nonlinearity_output).all().item():
    raise ValueError("Captured nonlinearity output contains NaN or Inf")

embedding_weight = readout_svd_model.embedding.weight.detach().float().cpu()
embedding_directions = embedding_weight[:, :relevant_dim].T
embedding_directions = F.normalize(embedding_directions, dim=1, eps=1e-12)
embedding_projection_values = embedding_directions @ captured_nonlinearity_output

right_singular_directions = F.normalize(readout_vh.detach().float().cpu(), dim=1, eps=1e-12)
right_singular_projection_values = right_singular_directions @ captured_nonlinearity_output

input_sequence_df = pd.DataFrame(
    {
        "position": list(range(input_dim)),
        "value": input_sequence.detach().cpu().to(torch.int64).numpy(),
        "is_relevant": [idx < relevant_dim for idx in range(input_dim)],
    }
)
embedding_projection_df = pd.DataFrame(
    {
        "input_position": list(range(relevant_dim)),
        "projection_on_embedding_direction": embedding_projection_values.numpy(),
    }
)
right_singular_projection_df = pd.DataFrame(
    {
        "singular_vector_idx": list(range(right_singular_projection_values.numel())),
        "singular_value": readout_singular_values.detach().float().cpu().numpy(),
        "projection_on_right_singular_vector": right_singular_projection_values.numpy(),
    }
)
output_df = pd.DataFrame(
    {
        "output_idx": list(range(full_output.numel())),
        "model_output": full_output.numpy(),
    }
)

prefix = (
    f"nonlinearity_projection_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}_"
    f"block_{NONLINEARITY_PROJECTION_BLOCK_IDX}"
)
input_sequence_path = ANALYSIS_DIR / f"{prefix}_input_sequence.csv"
embedding_projection_path = ANALYSIS_DIR / f"{prefix}_embedding_directions.csv"
right_singular_projection_path = ANALYSIS_DIR / f"{prefix}_right_singular_vectors.csv"
output_path = ANALYSIS_DIR / f"{prefix}_model_output.csv"
input_sequence_df.to_csv(input_sequence_path, index=False)
embedding_projection_df.to_csv(embedding_projection_path, index=False)
right_singular_projection_df.to_csv(right_singular_projection_path, index=False)
output_df.to_csv(output_path, index=False)

print(
    f"Captured nonlinearity output before residual addition in block "
    f"{NONLINEARITY_PROJECTION_BLOCK_IDX} with shape {tuple(captured_nonlinearity_output.shape)}."
)
print(f"Saved projection tables with prefix {prefix}")
print("Input sequence:")
display(input_sequence_df)
print("Model output from the full forward pass:")
display(output_df)
print("Projection of nonlinearity output onto first relevant embedding directions:")
display(embedding_projection_df)
print("Projection of nonlinearity output onto readout right singular vectors:")
display(right_singular_projection_df)


Apply the readout directly to one block's nonlinearity output and compare MSE against the ordinary full-model prediction.


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

from parity_net.analysis import per_degree_mse, resolve_test_data_path
from parity_net.data import ParityDataset, load_dataset, target_names

if "readout_svd_model" not in globals() or "readout_svd_payload" not in globals():
    raise NameError("Run the Readout Matrix SVD cell first")

READOUT_ON_NONLINEARITY_BLOCK_IDX = 0  # 0 means the first residual block.
READOUT_ON_NONLINEARITY_NUM_SAMPLES = 20_000  # Use None for the full saved test set.
READOUT_ON_NONLINEARITY_BATCH_SIZE = None  # Use None to reuse training batch_size.

readout_on_nonlinearity_config = readout_svd_payload["config"]
readout_on_nonlinearity_training = readout_on_nonlinearity_config["training"]
readout_on_nonlinearity_model_config = readout_on_nonlinearity_config["model"]
readout_on_nonlinearity_task_config = readout_on_nonlinearity_config.get("task") or {
    "input_dim": readout_on_nonlinearity_model_config["input_dim"],
    "relevant_dim": readout_on_nonlinearity_model_config["relevant_dim"],
    "exclude_targets": [],
}
readout_on_nonlinearity_target_names = target_names(
    int(readout_on_nonlinearity_task_config["relevant_dim"]),
    list(readout_on_nonlinearity_task_config.get("exclude_targets", [])),
)

num_blocks = len(readout_svd_model.blocks)
if READOUT_ON_NONLINEARITY_BLOCK_IDX < 0 or READOUT_ON_NONLINEARITY_BLOCK_IDX >= num_blocks:
    raise ValueError(f"READOUT_ON_NONLINEARITY_BLOCK_IDX must be in [0, {num_blocks - 1}]")

model_device = next(readout_svd_model.parameters()).device
model_dtype = next(readout_svd_model.parameters()).dtype
batch_size = READOUT_ON_NONLINEARITY_BATCH_SIZE or int(readout_on_nonlinearity_training["batch_size"])

test_data_path = resolve_test_data_path(
    readout_svd_checkpoint_path,
    readout_on_nonlinearity_training,
    readout_svd_payload.get("test_data_path"),
)
if test_data_path is None:
    raise FileNotFoundError("Could not find saved test_data.pt for this checkpoint")

test_data = load_dataset(test_data_path, model_device, model_dtype)
if READOUT_ON_NONLINEARITY_NUM_SAMPLES is not None:
    if READOUT_ON_NONLINEARITY_NUM_SAMPLES <= 0:
        raise ValueError("READOUT_ON_NONLINEARITY_NUM_SAMPLES must be positive or None")
    if READOUT_ON_NONLINEARITY_NUM_SAMPLES > test_data.x.shape[0]:
        raise ValueError(
            f"Requested {READOUT_ON_NONLINEARITY_NUM_SAMPLES} samples, but saved test set has "
            f"{test_data.x.shape[0]}"
        )
    test_data = ParityDataset(
        x=test_data.x[:READOUT_ON_NONLINEARITY_NUM_SAMPLES],
        y=test_data.y[:READOUT_ON_NONLINEARITY_NUM_SAMPLES],
    )

baseline_preds = []
nonlinearity_readout_preds = []
readout_svd_model.eval()
with torch.no_grad():
    for start in range(0, test_data.x.shape[0], batch_size):
        stop = min(start + batch_size, test_data.x.shape[0])
        x_batch = test_data.x[start:stop]
        baseline_preds.append(readout_svd_model(x_batch))

        h = readout_svd_model.embedding(x_batch)
        captured_nonlinearity = None
        for block_idx, block in enumerate(readout_svd_model.blocks):
            nonlinearity_output = block.activation(block.linear(h))
            if block_idx == READOUT_ON_NONLINEARITY_BLOCK_IDX:
                captured_nonlinearity = nonlinearity_output
                break
            if block.post_activation_linear is not None:
                residual_update = block.post_activation_linear(nonlinearity_output)
            else:
                residual_update = nonlinearity_output
            h = h + residual_update
        if captured_nonlinearity is None:
            raise RuntimeError("Failed to capture the requested nonlinearity output")
        readout_weight = readout_svd_model.readout_weight_matrix()
        readout_bias = readout_svd_model.readout_bias_vector()
        nonlinearity_readout_preds.append(F.linear(captured_nonlinearity, readout_weight, readout_bias))

baseline_pred = torch.cat(baseline_preds, dim=0)
nonlinearity_readout_pred = torch.cat(nonlinearity_readout_preds, dim=0)
if not torch.isfinite(baseline_pred).all().item():
    raise ValueError("Baseline predictions contain NaN or Inf")
if not torch.isfinite(nonlinearity_readout_pred).all().item():
    raise ValueError("Readout-on-nonlinearity predictions contain NaN or Inf")


def individual_target_mse_row(pred, y, target_names_):
    return {
        f"mse_{target_name}": F.mse_loss(pred[:, target_idx], y[:, target_idx]).item()
        for target_idx, target_name in enumerate(target_names_)
    }

comparison_rows = []
for prediction_name, pred in [
    ("full_model_output", baseline_pred),
    (f"readout_of_block_{READOUT_ON_NONLINEARITY_BLOCK_IDX}_nonlinearity", nonlinearity_readout_pred),
]:
    comparison_rows.append(
        {
            "N": READOUT_SVD_N,
            "checkpoint": readout_svd_checkpoint_label,
            "block_idx": READOUT_ON_NONLINEARITY_BLOCK_IDX,
            "prediction": prediction_name,
            "num_samples": test_data.x.shape[0],
            **per_degree_mse(pred, test_data.y, readout_on_nonlinearity_target_names),
            **individual_target_mse_row(pred, test_data.y, readout_on_nonlinearity_target_names),
        }
    )

readout_on_nonlinearity_mse_df = pd.DataFrame(comparison_rows)
output_path = (
    ANALYSIS_DIR
    / f"readout_on_nonlinearity_mse_N_{READOUT_SVD_N}_{readout_svd_checkpoint_label}_block_{READOUT_ON_NONLINEARITY_BLOCK_IDX}.csv"
)
readout_on_nonlinearity_mse_df.to_csv(output_path, index=False)

print(
    f"Compared ordinary model output against readout(phi(Wx)) at block "
    f"{READOUT_ON_NONLINEARITY_BLOCK_IDX} on {test_data.x.shape[0]} saved test samples."
)
print(f"Saved MSE comparison to {output_path}")
display(readout_on_nonlinearity_mse_df)


## Parity Directions in Nonlinearity Activations

Estimate vectors E_x[r(x) m(x)] for each available relevant d2 parity monomial, where r(x) is the selected residual block's nonlinearity output before residual addition.


In [ ]:
import pandas as pd
import torch

from parity_net.analysis import resolve_test_data_path
from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset, target_names
from parity_net.train import resolve_device, resolve_dtype

PARITY_DIRECTION_N = 2048
PARITY_DIRECTION_CHECKPOINT_STEP = "final"  # Use "final" or an integer step, e.g. 1000.
PARITY_DIRECTION_BLOCK_IDX = 0  # 0 means the first residual block.
PARITY_DIRECTION_NUM_SAMPLES = 20_000  # Use None for the full saved test set.
PARITY_DIRECTION_BATCH_SIZE = None  # Use None to reuse training batch_size.


def parity_direction_checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('PARITY_DIRECTION_CHECKPOINT_STEP must be "final" or an integer step')


parity_direction_checkpoint_file, parity_direction_checkpoint_label = parity_direction_checkpoint_name(
    PARITY_DIRECTION_CHECKPOINT_STEP
)
parity_direction_run_dir = RUNS_DIR / f"N_{PARITY_DIRECTION_N}"
parity_direction_checkpoint_path = parity_direction_run_dir / "checkpoints" / parity_direction_checkpoint_file
if not parity_direction_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {parity_direction_checkpoint_path}")

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
parity_direction_model, parity_direction_payload, _ = load_checkpoint(
    parity_direction_checkpoint_path,
    load_device,
)
parity_direction_config = parity_direction_payload["config"]
parity_direction_training = parity_direction_config["training"]
parity_direction_model_config = parity_direction_config["model"]
parity_direction_task_config = parity_direction_config.get("task") or {
    "input_dim": parity_direction_model_config["input_dim"],
    "relevant_dim": parity_direction_model_config["relevant_dim"],
    "exclude_targets": [],
}
parity_direction_target_names = target_names(
    int(parity_direction_task_config["relevant_dim"]),
    list(parity_direction_task_config.get("exclude_targets", [])),
)
parity_direction_d2_indices = [
    idx for idx, name in enumerate(parity_direction_target_names) if name.startswith("d2_")
]
parity_direction_d2_names = [parity_direction_target_names[idx] for idx in parity_direction_d2_indices]
if not parity_direction_d2_indices:
    raise ValueError("This task has no d2 parity targets available")

num_blocks = len(parity_direction_model.blocks)
if PARITY_DIRECTION_BLOCK_IDX < 0 or PARITY_DIRECTION_BLOCK_IDX >= num_blocks:
    raise ValueError(f"PARITY_DIRECTION_BLOCK_IDX must be in [0, {num_blocks - 1}]")

device = resolve_device(parity_direction_training["device"])
dtype = resolve_dtype(parity_direction_training["dtype"])
parity_direction_model = parity_direction_model.to(device=device, dtype=dtype).eval()
batch_size = PARITY_DIRECTION_BATCH_SIZE or int(parity_direction_training["batch_size"])

test_data_path = resolve_test_data_path(
    parity_direction_checkpoint_path,
    parity_direction_training,
    parity_direction_payload.get("test_data_path"),
)
if test_data_path is None:
    raise FileNotFoundError("Could not find saved test_data.pt for this checkpoint")

test_data = load_dataset(test_data_path, device, dtype)
if PARITY_DIRECTION_NUM_SAMPLES is not None:
    if PARITY_DIRECTION_NUM_SAMPLES <= 0:
        raise ValueError("PARITY_DIRECTION_NUM_SAMPLES must be positive or None")
    if PARITY_DIRECTION_NUM_SAMPLES > test_data.x.shape[0]:
        raise ValueError(
            f"Requested {PARITY_DIRECTION_NUM_SAMPLES} samples, but saved test set has "
            f"{test_data.x.shape[0]}"
        )
    test_data = ParityDataset(
        x=test_data.x[:PARITY_DIRECTION_NUM_SAMPLES],
        y=test_data.y[:PARITY_DIRECTION_NUM_SAMPLES],
    )

width = int(parity_direction_model_config["N"])
parity_direction_accumulator = torch.zeros(
    (len(parity_direction_d2_indices), width),
    device=device,
    dtype=torch.float64,
)
num_seen = 0
with torch.no_grad():
    for start in range(0, test_data.x.shape[0], batch_size):
        stop = min(start + batch_size, test_data.x.shape[0])
        x_batch = test_data.x[start:stop]
        y_d2_batch = test_data.y[start:stop, parity_direction_d2_indices].to(dtype=torch.float64)
        h = parity_direction_model.embedding(x_batch)
        captured_nonlinearity = None
        for block_idx, block in enumerate(parity_direction_model.blocks):
            nonlinearity_output = block.activation(block.linear(h))
            if block_idx == PARITY_DIRECTION_BLOCK_IDX:
                captured_nonlinearity = nonlinearity_output
                break
            if block.post_activation_linear is not None:
                residual_update = block.post_activation_linear(nonlinearity_output)
            else:
                residual_update = nonlinearity_output
            h = h + residual_update
        if captured_nonlinearity is None:
            raise RuntimeError("Failed to capture the requested nonlinearity output")
        if not torch.isfinite(captured_nonlinearity).all().item():
            raise ValueError("Captured nonlinearity activations contain NaN or Inf")
        parity_direction_accumulator += y_d2_batch.T @ captured_nonlinearity.to(dtype=torch.float64)
        num_seen += x_batch.shape[0]

parity_activation_direction_vectors = (parity_direction_accumulator / num_seen).detach().cpu()
parity_activation_direction_names = parity_direction_d2_names
parity_activation_direction_target_indices = parity_direction_d2_indices

parity_direction_vectors_df = pd.DataFrame(
    parity_activation_direction_vectors.T.numpy(),
    columns=parity_activation_direction_names,
)
parity_direction_vectors_df.insert(0, "feature_idx", range(parity_activation_direction_vectors.shape[1]))
parity_direction_vectors_path = (
    ANALYSIS_DIR
    / f"parity_activation_directions_N_{PARITY_DIRECTION_N}_{parity_direction_checkpoint_label}_block_{PARITY_DIRECTION_BLOCK_IDX}.csv"
)
parity_direction_vectors_df.to_csv(parity_direction_vectors_path, index=False)

print(
    f"Computed {len(parity_activation_direction_names)} d2 parity activation directions "
    f"from {num_seen} saved test samples at block {PARITY_DIRECTION_BLOCK_IDX}."
)
print(f"Saved direction vectors to {parity_direction_vectors_path}")
display(parity_direction_vectors_df.head())


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

if "parity_activation_direction_vectors" not in globals():
    raise NameError("Run the parity-direction estimation cell first")

parity_direction_norms = torch.linalg.vector_norm(parity_activation_direction_vectors, dim=1)
normalized_parity_directions = F.normalize(parity_activation_direction_vectors, dim=1, eps=1e-12)
parity_direction_cosine_matrix = normalized_parity_directions @ normalized_parity_directions.T

parity_direction_norms_df = pd.DataFrame(
    {
        "parity": parity_activation_direction_names,
        "norm": parity_direction_norms.numpy(),
    }
)
parity_direction_cosine_df = pd.DataFrame(
    parity_direction_cosine_matrix.numpy(),
    index=parity_activation_direction_names,
    columns=parity_activation_direction_names,
)

prefix = f"parity_activation_direction_N_{PARITY_DIRECTION_N}_{parity_direction_checkpoint_label}_block_{PARITY_DIRECTION_BLOCK_IDX}"
norms_path = ANALYSIS_DIR / f"{prefix}_norms.csv"
cosines_path = ANALYSIS_DIR / f"{prefix}_cosines.csv"
parity_direction_norms_df.to_csv(norms_path, index=False)
parity_direction_cosine_df.to_csv(cosines_path)

print(f"Saved direction norms to {norms_path}")
print(f"Saved direction cosine matrix to {cosines_path}")
print("Activation-direction norms:")
display(parity_direction_norms_df)
print("Activation-direction cosine similarity matrix:")
display(parity_direction_cosine_df)


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F

if "parity_activation_direction_vectors" not in globals():
    raise NameError("Run the parity-direction estimation cell first")

readout_vectors_for_parities = (
    parity_direction_model.readout_weight_matrix().detach().float().cpu()[parity_activation_direction_target_indices]
)
normalized_readout_vectors = F.normalize(readout_vectors_for_parities, dim=1, eps=1e-12)
normalized_parity_directions = F.normalize(
    parity_activation_direction_vectors.to(dtype=normalized_readout_vectors.dtype),
    dim=1,
    eps=1e-12,
)

readout_cosine_matrix = normalized_readout_vectors @ normalized_readout_vectors.T
activation_readout_cosine_matrix = normalized_parity_directions @ normalized_readout_vectors.T
corresponding_activation_readout_cosines = torch.diag(activation_readout_cosine_matrix)

readout_cosine_df = pd.DataFrame(
    readout_cosine_matrix.numpy(),
    index=parity_activation_direction_names,
    columns=parity_activation_direction_names,
)
activation_readout_cosine_df = pd.DataFrame(
    activation_readout_cosine_matrix.numpy(),
    index=[f"activation_direction_{name}" for name in parity_activation_direction_names],
    columns=[f"readout_{name}" for name in parity_activation_direction_names],
)
corresponding_activation_readout_df = pd.DataFrame(
    {
        "parity": parity_activation_direction_names,
        "cosine_activation_direction_with_matching_readout": corresponding_activation_readout_cosines.numpy(),
    }
)

prefix = f"parity_activation_readout_N_{PARITY_DIRECTION_N}_{parity_direction_checkpoint_label}_block_{PARITY_DIRECTION_BLOCK_IDX}"
readout_cosines_path = ANALYSIS_DIR / f"{prefix}_readout_cosines.csv"
activation_readout_cosines_path = ANALYSIS_DIR / f"{prefix}_activation_readout_cosines.csv"
corresponding_path = ANALYSIS_DIR / f"{prefix}_matching_cosines.csv"
readout_cosine_df.to_csv(readout_cosines_path)
activation_readout_cosine_df.to_csv(activation_readout_cosines_path)
corresponding_activation_readout_df.to_csv(corresponding_path, index=False)

print(f"Saved readout-vector cosine matrix to {readout_cosines_path}")
print(f"Saved activation-direction/readout cosine matrix to {activation_readout_cosines_path}")
print(f"Saved matching activation-direction/readout cosines to {corresponding_path}")
print("Readout-vector cosine similarity matrix:")
display(readout_cosine_df)
print("Activation directions vs corresponding readout vectors:")
display(corresponding_activation_readout_df)
print("Full activation-direction/readout cosine similarity matrix:")
display(activation_readout_cosine_df)
